In [ ]:
import torch
import torchvision
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from entrega_2.utils.chess_dataset import ChessDataset, get_transforms, collate_fn
from entrega_2.utils.testing import ObjectDetectionEvaluator
from entrega_2.utils.training import ObjectDetectionTrainer, plot_training_history

In [ ]:
TRAIN_IMG_DIR = "../../data/chess-pieces-coco/train"
TRAIN_ANN_FILE = "../../data/chess-pieces-coco/train/_annotations.coco.json"
VAL_IMG_DIR = "../../data/chess-pieces-coco/valid"
VAL_ANN_FILE = "../../data/chess-pieces-coco/valid/_annotations.coco.json"
TEST_IMG_DIR = "../../data/chess-pieces-coco/test"
TEST_ANN_FILE = "../../data/chess-pieces-coco/test/_annotations.coco.json"
NUM_CLASES = 13 # 12 piezas de ajedrez + 1 fondo = 13 clases totales

In [ ]:
train_dataset = ChessDataset(root=TRAIN_IMG_DIR, annFile=TRAIN_ANN_FILE, transforms=get_transforms(train=True))
val_dataset = ChessDataset(root=VAL_IMG_DIR, annFile=VAL_ANN_FILE, transforms=get_transforms(train=False))
test_dataset = ChessDataset(root=TEST_IMG_DIR, annFile=TEST_ANN_FILE, transforms=get_transforms(train=False))

In [ ]:
train_loader = DataLoader(
    train_dataset,
    batch_size=4,
    shuffle=True,
    num_workers=2,
    collate_fn=collate_fn
)

val_loader = DataLoader(
    val_dataset,
    batch_size=4,
    shuffle=False,
    num_workers=2,
    collate_fn=collate_fn
)

test_loader = DataLoader(
    test_dataset,
    batch_size=4,
    shuffle=False,
    num_workers=2,
    collate_fn=collate_fn
)

print(f"Batches de entrenamiento: {len(train_loader)} | Batches de validación: {len(val_loader)} | Batches de pruebas: {len(test_loader)}")

In [ ]:
model = torchvision.models.detection.fasterrcnn_resnet50_fpn_v2(
    weights="DEFAULT",
    min_size = 416,
    max_size = 416
)

in_features = model.roi_heads.box_predictor.cls_score.in_features

model.roi_heads.box_predictor = FastRCNNPredictor(in_features, NUM_CLASES)
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
model.to(device)

print(f"Modelo cargado y enviado a: {device}")

In [ ]:
params = [p for p in model.parameters() if p.requires_grad]
optimizer = optim.SGD(params, lr=0.01, momentum=0.93, weight_decay=0.0001)

lr_scheduler = torch.optim.lr_scheduler.StepLR(optimizer=optimizer, step_size=3, gamma=0.1)

trainer = ObjectDetectionTrainer(
    model=model,
    optimizer=optimizer,
    device=device,
    lr_scheduler=None,
    patience=5,
    save_path="mejor_faster_rcnn.pth"
)

In [ ]:
historial = trainer.fit(train_loader, val_loader, epochs=50)

In [ ]:
plot_training_history(historial)

In [ ]:
state_dict = torch.load("mejor_faster_rcnn.pth", weights_only=True)
model.load_state_dict(state_dict)

In [ ]:
CHESS_CLASSES = {
    1:  "Alfil Negro",      # black-bishop
    2:  "Rey Negro",        # black-king
    3:  "Caballo Negro",    # black-knight
    4:  "Peon Negro",       # black-pawn
    5:  "Reina Negra",      # black-queen
    6:  "Torre Negra",      # black-rook
    7:  "Alfil Blanco",     # white-bishop
    8:  "Rey Blanco",       # white-king
    9:  "Caballo Blanco",   # white-knight
    10: "Peon Blanco",      # white-pawn
    11: "Reina Blanca",     # white-queen
    12: "Torre Blanca"      # white-rook
}

In [ ]:
evaluador = ObjectDetectionEvaluator(model=model, device=device, class_names=CHESS_CLASSES)

In [ ]:
resultados_finales = evaluador.evaluate(test_loader)

In [ ]:
evaluador.plot_global_metrics(resultados_finales)
evaluador.plot_per_class_map(resultados_finales)

In [ ]:
# Ver predicciones reales en imágenes aleatorias contrastadas con sus etiquetas originales
evaluador.visualize_predictions(test_loader, num_images=3, score_threshold=0.6)